In [1]:
import sys, time
sys.path.append("..")
from functools import partial
import numpy as np
import pandas as pd
from src.data import load_cuad, TARGET_CLAUSES
from src.split import split_contracts
from src.chunking import clause_chunks
from src.embed import cached_chunk_embeddings, embed_query
from src.queries import CLAUSE_QUERIES
from src.metrics import evaluate_ranker
from src.db import connect, init_schema, ingest_contract, make_pgvector_ranker, search_contract

contracts = load_cuad()
dev_idx, test_idx = split_contracts(contracts)
chunker = partial(clause_chunks, max_size=1500)
tag = "clause-200-1500-200"

conn = connect()
init_schema(conn)
for i in dev_idx:
    c = contracts[i]
    chunks = chunker(c["text"])
    vecs = cached_chunk_embeddings(c["title"], chunks, tag)
    ingest_contract(conn, c["title"], tag, chunks, vecs)

print(conn.execute("SELECT count(*) FROM chunks").fetchone())

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

(21132,)


In [2]:
pg = make_pgvector_ranker(conn, tag)
rows = {cl: evaluate_ranker(pg, contracts, dev_idx, cl) for cl in TARGET_CLAUSES}
pg_res = pd.DataFrame(rows).T

ref = pd.read_csv("../eval/results_dense_clause1500_dev.csv", index_col=0)
cols = ["hit@1", "hit@3", "mrr"]
(pg_res[cols] - ref[cols]).round(3)

,hit@1,hit@3,mrr
Governing Law,0.0,0.0,0.0
Renewal Term,0.0,0.0,0.0
Notice Period To Terminate Renewal,0.0,0.0,0.0
Termination For Convenience,0.0,0.0,0.0
Non-Compete,0.0,0.0,0.0
Cap On Liability,0.0,0.0,0.0
Uncapped Liability,0.0,0.0,0.0


In [3]:
c = contracts[dev_idx[0]]
q = embed_query(CLAUSE_QUERIES["Governing Law"]).astype(np.float32)

times = []
for _ in range(200):
    t = time.perf_counter()
    search_contract(conn, c["title"], tag, q, k=10)
    times.append((time.perf_counter() - t) * 1000)
print("median ms:", round(float(np.median(times)), 2), "| p95 ms:", round(float(np.percentile(times, 95)), 2))

median ms: 1.13 | p95 ms: 1.57


In [4]:
queries = {cl: embed_query(CLAUSE_QUERIES[cl]).astype(np.float32) for cl in TARGET_CLAUSES}

def search_all(q, k=10):
    rows = conn.execute(
        "SELECT contract_title, start_char FROM chunks WHERE tag = %s "
        "ORDER BY embedding <=> %s LIMIT %s",
        (tag, q, k),
    ).fetchall()
    return [(t, s) for t, s in rows]

def timed(fn):
    times = []
    for _ in range(30):
        for q in queries.values():
            t = time.perf_counter()
            fn(q)
            times.append((time.perf_counter() - t) * 1000)
    return round(float(np.median(times)), 2)

exact = {cl: search_all(q) for cl, q in queries.items()}
print("exact scan median ms:", timed(search_all))

conn.execute("CREATE INDEX IF NOT EXISTS chunks_hnsw ON chunks USING hnsw (embedding vector_cosine_ops)")
print("hnsw median ms:", timed(search_all))

recalls = [len(set(search_all(q)) & set(exact[cl])) / 10 for cl, q in queries.items()]
print("hnsw recall@10 vs exact:", round(sum(recalls) / len(recalls), 3))

exact scan median ms: 39.09
hnsw median ms: 1.1
hnsw recall@10 vs exact: 1.0
